# ARC v0.21 — E5 RASF Cross-Encoder Replication

**Purpose.** Replicate the Risk-Aware Selective Fidelity (RASF) pipeline under the frozen FEVER E5 representation-approximation setting.

This is a **new post-v0.20 extension**. It does not alter any earlier sealed H1–H4, v0.18 encoder-transfer, v0.19 nprobe, v0.20c HNSW, or v0.20d HNSW-severity claim gate.

The design preserves the original RASF logic:

- search/evaluation stays on the lower-fidelity retriever (PQ32);
- only the feedback source is selectively upgraded to SQ8;
- risk prediction uses **only lower-fidelity observable features** plus policy metadata;
- the predictor is fit on the deterministic FEVER FIT split and applied unchanged to untouched validation;
- primary intervention budget = **25%**;
- secondary budgets = **10% and 50%**;
- primary policy configurations are reused from the prior BGE RASF closure:
  - mean: `alpha=0.3, k=20`;
  - softmax: `alpha=0.5, k=5, tau=0.1`.

The E5 representation setting is intentionally challenging because amplification prevalence is much higher than under BGE. The main question is therefore:

> Does risk-aware selective feedback still outperform matched-budget random allocation after the encoder and prevalence regime change?

### Evidence boundary

This notebook is a **cross-encoder method replication**, not a zero-shot model-transfer experiment. The logistic selector is re-fit on **E5 FIT** using the same deployable feature family and then frozen for E5 validation.

No FEVER test qrels are accessed.


In [ ]:
%pip install -q faiss-cpu==1.12.0 pyarrow psutil scikit-learn==1.7.1

from pathlib import Path
import os, json, hashlib, math, time, gc, sys
import numpy as np
import pandas as pd
import faiss
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

print("Python :", sys.version.split()[0])
print("FAISS  :", faiss.__version__)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("RAM GiB:", round(psutil.virtual_memory().total/2**30, 2))

assert faiss.__version__ == "1.12.0"
print("ENVIRONMENT CHECK: PASS")


## 1. Frozen replication configuration

In [ ]:
CONFIG = {
    "study_id": "ARC-v0.21-E5-RASF-CROSS-ENCODER-REPLICATION",
    "seed": 20260821,

    "encoder": "intfloat/e5-small-v2",
    "dim": 384,
    "nprobe": 64,
    "top_k": 100,
    "utility_k": 10,
    "rounds": 4,
    "epsilon": 0.002,

    # Same deployable predictor family as the prior RASF line.
    "logistic_C": 0.5,
    "class_weight": "balanced",
    "max_iter": 5000,

    # Primary + secondary budgets fixed before E5 validation intervention outcomes.
    "primary_budget": 0.25,
    "budgets": [0.10, 0.25, 0.50],

    # Exact policy configurations reused from prior BGE RASF closure.
    "policies": [
        {
            "name": "mean-k20-a0p3",
            "family": "mean",
            "alpha": 0.3,
            "k": 20,
            "temperature": None,
        },
        {
            "name": "softmax-k5-a0p5-t0p1",
            "family": "softmax",
            "alpha": 0.5,
            "k": 5,
            "temperature": 0.1,
        },
    ],

    "bootstrap_reps": 2000,
    "random_allocations": 10000,
    "checkpoint_every_queries": 50,
}
print(json.dumps(CONFIG, indent=2))


## 2. Mount Drive and resolve frozen E5 lineage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints")
ARC_ROOT = DRIVE_ROOT / "arc-v0"

V018 = ARC_ROOT / "cross-encoder-fever-replication-v018" / "20260819-015645"
V018_PROTOCOL = V018 / "v018_cross_encoder_protocol.json"
V018_MANIFEST = V018 / "corpus_encoding_manifest.json"
V018_QUERY_EMB = V018 / "dev_query_embeddings.float32.npy"
V018_QUERY_IDS = V018 / "dev_query_ids.txt"
V018_FIT_ENDPOINTS = V018 / "v018_fit_endpoints.parquet"
V018_VAL_ENDPOINTS = V018 / "v018_validation_endpoints.parquet"
V018_PQ = V018 / "fever-e5-small-v2-ivfpq-nlist4096-m32-nbits8.faiss"
V018_SQ = V018 / "fever-e5-small-v2-ivfsq8-nlist4096.faiss"
V018_SHARDS = V018 / "corpus_shards"

V013_SPLIT_CANDIDATES = [
    ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-151852" / "v013_boundary_query_split.csv",
    ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-140640" / "v013_boundary_query_split.csv",
]
V013_SPLIT = next((p for p in V013_SPLIT_CANDIDATES if p.exists()), None)

FEVER_QRELS_DEV = DRIVE_ROOT / "raw-datasets" / "fever" / "qrels" / "dev.tsv"

OUT = ARC_ROOT / "e5-rasf-cross-encoder-replication-v021"
OUT.mkdir(parents=True, exist_ok=True)

required = [
    V018_PROTOCOL, V018_MANIFEST, V018_QUERY_EMB, V018_QUERY_IDS,
    V018_FIT_ENDPOINTS, V018_VAL_ENDPOINTS, V018_PQ, V018_SQ,
    V018_SHARDS, V013_SPLIT, FEVER_QRELS_DEV
]
missing = [str(p) for p in required if p is None or not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(missing))

print("V018 :", V018)
print("OUT  :", OUT)
print("DRIVE PATH CHECK: PASS")


## 3. Provenance audit and v0.21 protocol freeze

In [ ]:
def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def read_id_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

protocol = json.loads(V018_PROTOCOL.read_text())
manifest = json.loads(V018_MANIFEST.read_text())

assert protocol["encoder"] == CONFIG["encoder"]
assert int(protocol["dimension"]) == CONFIG["dim"]
assert int(protocol["feedback"]["rounds"]) == CONFIG["rounds"]
assert int(protocol["feedback"]["top_retrieve"]) == CONFIG["top_k"]
assert int(protocol["feedback"]["utility_k"]) == CONFIG["utility_k"]
assert float(protocol["epsilon_primary"]) == CONFIG["epsilon"]
assert protocol["test_accessed"] is False
assert protocol["test_relevance_accessed"] is False
assert int(manifest["rows"]) == 5_416_568
assert len(manifest["shards"]) == 55

V021_PROTOCOL = {
    "study_id": CONFIG["study_id"],
    "status": "E5_RASF_PROTOCOL_FROZEN_BEFORE_VALIDATION_INTERVENTION_OUTCOMES",
    "source_v018_protocol_sha256": sha256_file(V018_PROTOCOL),
    "source_v018_manifest_sha256": sha256_file(V018_MANIFEST),
    "source_v018_fit_endpoints_sha256": sha256_file(V018_FIT_ENDPOINTS),
    "source_v018_validation_endpoints_sha256": sha256_file(V018_VAL_ENDPOINTS),
    "encoder": CONFIG["encoder"],
    "retriever_pair": ["IVF-PQ32", "IVF-SQ8"],
    "nprobe": CONFIG["nprobe"],
    "feedback_policies": CONFIG["policies"],
    "primary_budget": CONFIG["primary_budget"],
    "secondary_budgets": CONFIG["budgets"],
    "risk_features": [
        "pq_score_entropy_top100",
        "pq_top1_minus_top10_margin",
        "alpha",
        "log_k",
        "feedback_family_indicator",
        "numeric_temperature",
    ],
    "risk_target": "1[H3abs > 0.002]",
    "selector": {
        "model": "StandardScaler + LogisticRegression",
        "C": CONFIG["logistic_C"],
        "class_weight": CONFIG["class_weight"],
        "max_iter": CONFIG["max_iter"],
    },
    "intervention": (
        "Search and evaluation stay on PQ32. "
        "Selected queries receive SQ8-sourced feedback at every feedback round."
    ),
    "fit_membership_sha256": protocol["fit_membership_sha256"],
    "validation_membership_sha256": protocol["validation_membership_sha256"],
    "random_allocations": CONFIG["random_allocations"],
    "test_accessed": False,
    "test_relevance_accessed": False,
}

PROTOCOL_PATH = OUT / "V021_E5_RASF_PROTOCOL.json"
if PROTOCOL_PATH.exists():
    old = json.loads(PROTOCOL_PATH.read_text())
    assert old["feedback_policies"] == V021_PROTOCOL["feedback_policies"]
    assert old["primary_budget"] == V021_PROTOCOL["primary_budget"]
else:
    PROTOCOL_PATH.write_text(json.dumps(V021_PROTOCOL, indent=2))

print(json.dumps(V021_PROTOCOL, indent=2))
print("V0.21 PROTOCOL FROZEN")


## 4. Load split, queries, qrels, and E5 endpoint tables

In [ ]:
def canonical_membership_hash(ids):
    return hashlib.sha256(
        "\n".join(sorted(map(str, ids))).encode("utf-8")
    ).hexdigest()

queries = np.load(V018_QUERY_EMB, mmap_mode="r")
query_ids = np.asarray(read_id_lines(V018_QUERY_IDS), dtype=object)
assert queries.shape == (6666, 384)
assert len(query_ids) == 6666

split_df = pd.read_csv(V013_SPLIT)
split_df["query_id"] = split_df["query_id"].astype(str)
labels = split_df["split"].astype(str).str.lower()
fit_ids = set(split_df.loc[labels=="fit", "query_id"])
val_ids = set(split_df.loc[labels=="validation", "query_id"])
assert len(fit_ids) == 3350
assert len(val_ids) == 3316
assert fit_ids.isdisjoint(val_ids)
assert canonical_membership_hash(fit_ids) == protocol["fit_membership_sha256"]
assert canonical_membership_hash(val_ids) == protocol["validation_membership_sha256"]

fit_ep = pd.read_parquet(V018_FIT_ENDPOINTS)
val_ep = pd.read_parquet(V018_VAL_ENDPOINTS)

# Normalize likely naming differences.
def normalize_endpoint_columns(df):
    z = df.copy()
    rename = {}
    for c in z.columns:
        lc = c.lower()
        if lc in {"method", "feedback_method"}:
            rename[c] = "family"
        elif lc in {"tau", "temp"}:
            rename[c] = "temperature"
        elif lc in {"h3", "h3_abs", "h3_abs_slope", "h3abs_slope"}:
            rename[c] = "H3abs"
        elif lc in {"qid", "queryid"}:
            rename[c] = "query_id"
    z = z.rename(columns=rename)
    if "H3abs" not in z.columns:
        candidates = [c for c in z.columns if "h3" in c.lower() and "abs" in c.lower()]
        if len(candidates) == 1:
            z = z.rename(columns={candidates[0]:"H3abs"})
    required = {"query_id","family","alpha","k","H3abs"}
    if not required <= set(z.columns):
        raise ValueError(f"Endpoint columns missing {required-set(z.columns)}; got {list(z.columns)}")
    z["query_id"] = z["query_id"].astype(str)
    return z

fit_ep = normalize_endpoint_columns(fit_ep)
val_ep = normalize_endpoint_columns(val_ep)

qrels = pd.read_csv(FEVER_QRELS_DEV, sep="\t")
rename = {}
for c in qrels.columns:
    s = c.lower().replace("_","-")
    if s in {"query-id","queryid","qid"}:
        rename[c] = "query_id"
    elif s in {"corpus-id","corpusid","doc-id","docid"}:
        rename[c] = "doc_id"
    elif s in {"score","relevance","rel"}:
        rename[c] = "relevance"
qrels = qrels.rename(columns=rename)
if not {"query_id","doc_id","relevance"} <= set(qrels.columns):
    qrels = pd.read_csv(
        FEVER_QRELS_DEV, sep="\t",
        names=["query_id","doc_id","relevance"]
    )
qrels["query_id"] = qrels["query_id"].astype(str)
qrels["doc_id"] = qrels["doc_id"].astype(str)
qrels["relevance"] = qrels["relevance"].astype(float)

QREL_MAP = {
    str(qid): {str(d):float(r) for d,r in zip(sub["doc_id"], sub["relevance"])}
    for qid, sub in qrels.groupby("query_id")
}

print("FIT endpoints:", fit_ep.shape)
print("VAL endpoints:", val_ep.shape)
print("INPUT AUDIT: PASS")


## 5. Load E5 PQ32 / SQ8 indexes and construct document-ID sidecar if needed

In [ ]:
pq_index = faiss.read_index(str(V018_PQ))
sq_index = faiss.read_index(str(V018_SQ))
pq_index.nprobe = CONFIG["nprobe"]
sq_index.nprobe = CONFIG["nprobe"]

assert pq_index.ntotal == 5_416_568
assert sq_index.ntotal == 5_416_568

DOC_IDS_PATH = OUT / "fever_e5_doc_ids.txt"

if not DOC_IDS_PATH.exists():
    with open(DOC_IDS_PATH, "w", encoding="utf-8") as out_ids:
        total = 0
        for sid in range(55):
            p = V018_SHARDS / f"shard-{sid:04d}.ids.txt"
            ids = read_id_lines(p)
            out_ids.write("\n".join(ids) + "\n")
            total += len(ids)
    assert total == 5_416_568

doc_ids = np.asarray(read_id_lines(DOC_IDS_PATH), dtype=object)
assert len(doc_ids) == 5_416_568

print("PQ/SQ indexes loaded.")
print("PQ ntotal:", pq_index.ntotal, "SQ ntotal:", sq_index.ntotal)


## 6. Lower-fidelity observable query features

For each initial query, retrieve top-100 with E5 PQ32 only.

Two query-specific features are computed:

1. **score entropy**: Shannon entropy of the softmax-normalized PQ32 top-100 score vector;
2. **top-1/top-10 margin**: `score[0] - score[9]`.

No qrels, SQ8 result, cross-retriever divergence, or future trajectory quantity enters the predictor.


In [ ]:
def query_mask(ids, selected):
    selected = set(map(str, selected))
    return np.asarray([str(x) in selected for x in ids], dtype=bool)

def score_entropy(scores):
    s = np.asarray(scores, dtype=np.float64)
    z = s - s.max()
    p = np.exp(z)
    p /= p.sum()
    return float(-(p * np.log(np.maximum(p, 1e-15))).sum())

FEATURE_PATH = OUT / "v021_e5_pq_initial_query_features.parquet"

if FEATURE_PATH.exists():
    query_features = pd.read_parquet(FEATURE_PATH)
else:
    pq_index.nprobe = CONFIG["nprobe"]
    scores, ids = pq_index.search(np.asarray(queries, dtype=np.float32), CONFIG["top_k"])
    rows = []
    for i, qid in enumerate(query_ids):
        rows.append({
            "query_id": str(qid),
            "pq_score_entropy_top100": score_entropy(scores[i]),
            "pq_top1_minus_top10_margin": float(scores[i,0] - scores[i,9]),
        })
    query_features = pd.DataFrame(rows)
    query_features.to_parquet(FEATURE_PATH, index=False)

assert query_features["query_id"].nunique() == 6666
display(query_features.describe(include="all"))


## 7. Construct FIT/validation predictor matrices from the frozen 44-policy endpoints

In [ ]:
FEATURE_COLS = [
    "pq_score_entropy_top100",
    "pq_top1_minus_top10_margin",
    "alpha",
    "log_k",
    "feedback_family_indicator",
    "numeric_temperature",
]

def build_predictor_table(ep):
    z = ep.merge(query_features, on="query_id", how="left", validate="many_to_one")
    if z[["pq_score_entropy_top100","pq_top1_minus_top10_margin"]].isna().any().any():
        raise ValueError("Missing query features after merge.")

    z["family"] = z["family"].astype(str).str.lower()
    z["log_k"] = np.log(z["k"].astype(float))
    z["feedback_family_indicator"] = (z["family"]=="softmax").astype(float)

    if "temperature" not in z.columns:
        z["temperature"] = np.nan
    z["numeric_temperature"] = z["temperature"].fillna(0.0).astype(float)

    z["target"] = (z["H3abs"].astype(float) > CONFIG["epsilon"]).astype(int)
    return z

fit_pred = build_predictor_table(fit_ep)
val_pred = build_predictor_table(val_ep)

print("FIT prevalence:", fit_pred["target"].mean())
print("VAL prevalence:", val_pred["target"].mean())
print("FIT events:", len(fit_pred), "VAL events:", len(val_pred))


## 8. Fit E5 deployable selector on FIT only; freeze before validation intervention

In [ ]:
X_fit = fit_pred[FEATURE_COLS].to_numpy(dtype=np.float64)
y_fit = fit_pred["target"].to_numpy(dtype=int)
X_val = val_pred[FEATURE_COLS].to_numpy(dtype=np.float64)
y_val = val_pred["target"].to_numpy(dtype=int)

scaler = StandardScaler()
X_fit_s = scaler.fit_transform(X_fit)

model = LogisticRegression(
    C=CONFIG["logistic_C"],
    class_weight=CONFIG["class_weight"],
    max_iter=CONFIG["max_iter"],
    random_state=CONFIG["seed"],
)
model.fit(X_fit_s, y_fit)

fit_pred["risk_score"] = model.predict_proba(X_fit_s)[:,1]
val_pred["risk_score"] = model.predict_proba(scaler.transform(X_val))[:,1]

MODEL_FREEZE = {
    "study_id": CONFIG["study_id"],
    "fit_only": True,
    "feature_cols": FEATURE_COLS,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "logistic_coef": model.coef_[0].tolist(),
    "logistic_intercept": model.intercept_.tolist(),
    "C": CONFIG["logistic_C"],
    "class_weight": CONFIG["class_weight"],
    "max_iter": CONFIG["max_iter"],
    "fit_membership_sha256": protocol["fit_membership_sha256"],
    "validation_membership_sha256": protocol["validation_membership_sha256"],
    "validation_intervention_outcomes_inspected_before_freeze": False,
}
MODEL_FREEZE_PATH = OUT / "V021_E5_RISK_MODEL_FREEZE.json"
MODEL_FREEZE_PATH.write_text(json.dumps(MODEL_FREEZE, indent=2))

fit_pred.to_parquet(OUT/"v021_e5_fit_risk_events.parquet", index=False)
val_pred.to_parquet(OUT/"v021_e5_validation_risk_events.parquet", index=False)

print("E5 RISK MODEL FROZEN")
print(json.dumps(MODEL_FREEZE, indent=2)[:4000])


## 9. Predictor evaluation on untouched E5 validation

In [ ]:
def capture_at_fraction(y, score, frac):
    n = len(y)
    k = max(1, int(round(frac*n)))
    order = np.argsort(-np.asarray(score))
    denom = np.sum(y)
    if denom == 0:
        return np.nan
    return float(np.sum(np.asarray(y)[order[:k]]) / denom)

roc = roc_auc_score(y_val, val_pred["risk_score"])
pr = average_precision_score(y_val, val_pred["risk_score"])
cap25 = capture_at_fraction(y_val, val_pred["risk_score"], 0.25)

pred_point = {
    "roc_auc": float(roc),
    "pr_auc": float(pr),
    "capture_at_25pct": float(cap25),
    "prevalence": float(y_val.mean()),
}
print(json.dumps(pred_point, indent=2))

# Query-cluster bootstrap: sample queries and retain all 44 policy realizations.
def predictor_cluster_bootstrap(df, reps=2000, seed=20260821):
    qids = df["query_id"].drop_duplicates().to_numpy()
    grouped = {q: df[df["query_id"]==q].copy() for q in qids}
    rng = np.random.default_rng(seed)
    rows = []
    n = len(qids)

    for b in range(reps):
        sampled = qids[rng.integers(0,n,size=n)]
        boot = pd.concat([grouped[q] for q in sampled], ignore_index=True)
        y = boot["target"].to_numpy()
        s = boot["risk_score"].to_numpy()
        if len(np.unique(y)) < 2:
            continue
        rows.append({
            "roc_auc": roc_auc_score(y,s),
            "pr_auc": average_precision_score(y,s),
            "capture_at_25pct": capture_at_fraction(y,s,0.25),
        })

    bdf = pd.DataFrame(rows)
    return {
        m: {
            "low": float(bdf[m].quantile(0.025)),
            "high": float(bdf[m].quantile(0.975)),
        }
        for m in bdf.columns
    }

pred_ci = predictor_cluster_bootstrap(
    val_pred, CONFIG["bootstrap_reps"], CONFIG["seed"]
)
print(json.dumps(pred_ci, indent=2))


## 10. Select risk scores for the two frozen intervention configurations

Within a fixed configuration, policy metadata are constant across queries, so ranking is driven by the two lower-fidelity query score features.


In [ ]:
def select_config_rows(df, policy):
    fam = df["family"].astype(str).str.lower() == policy["family"]
    aa = np.isclose(df["alpha"].astype(float), policy["alpha"])
    kk = df["k"].astype(int) == policy["k"]

    if policy["temperature"] is None:
        if "temperature" in df.columns:
            tt = df["temperature"].isna() | np.isclose(df["temperature"].fillna(0), 0)
        else:
            tt = np.ones(len(df), dtype=bool)
    else:
        tt = np.isclose(df["temperature"].astype(float), policy["temperature"])

    z = df[fam & aa & kk & tt].copy()
    if z["query_id"].nunique() != 3316:
        raise ValueError(
            f"{policy['name']}: expected 3316 validation query rows; "
            f"got {z['query_id'].nunique()}"
        )
    return z[["query_id","risk_score"]].drop_duplicates("query_id")

RISK_BY_POLICY = {
    p["name"]: select_config_rows(val_pred, p)
    for p in CONFIG["policies"]
}
for name,z in RISK_BY_POLICY.items():
    print(name, z.shape, z["risk_score"].min(), z["risk_score"].max())


## 11. Feedback/evaluation implementation

In [ ]:
def normalize_vec(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    n = float(np.linalg.norm(x))
    return x/max(n,eps)

def dcg(rels):
    rels = np.asarray(rels,dtype=np.float64)
    if len(rels)==0:
        return 0.0
    gains = np.power(2.0, rels)-1.0
    discounts = 1.0/np.log2(np.arange(2,len(rels)+2))
    return float(np.sum(gains*discounts))

def ndcg_at_k(retrieved_doc_ids, qrel_dict, k=10):
    rels = [qrel_dict.get(str(d),0.0) for d in retrieved_doc_ids[:k]]
    ideal = sorted(qrel_dict.values(), reverse=True)[:k]
    denom = dcg(ideal)
    return 0.0 if denom==0 else dcg(rels)/denom

_SHARD_CACHE = {}

def get_corpus_rows(rows):
    rows = np.asarray(rows,dtype=np.int64)
    out = np.empty((len(rows),384),dtype=np.float32)
    by_shard = {}
    for oi,row in enumerate(rows):
        if row < 5_400_000:
            sid,off = int(row//100_000), int(row%100_000)
        else:
            sid,off = 54, int(row-5_400_000)
        by_shard.setdefault(sid,[]).append((oi,off))

    for sid,pairs in by_shard.items():
        if sid not in _SHARD_CACHE:
            _SHARD_CACHE[sid] = np.load(
                V018_SHARDS/f"shard-{sid:04d}.float16.npy",
                mmap_mode="r"
            )
        mm = _SHARD_CACHE[sid]
        outs = [a for a,_ in pairs]
        offs = [b for _,b in pairs]
        out[outs] = np.asarray(mm[offs],dtype=np.float32)
    return out

def feedback_vector(vecs,scores,policy):
    x = np.asarray(vecs,dtype=np.float32)
    if policy["family"]=="mean":
        f = x.mean(axis=0)
    else:
        z = np.asarray(scores,dtype=np.float64)/float(policy["temperature"])
        z -= z.max()
        w = np.exp(z)
        w /= w.sum()
        f = (x*w[:,None]).sum(axis=0)
    return normalize_vec(f)

def update_state(q0,f,alpha):
    return normalize_vec((1.0-alpha)*q0 + alpha*f)

def search(index,q):
    index.nprobe = CONFIG["nprobe"]
    return index.search(
        np.asarray(q,dtype=np.float32),
        CONFIG["top_k"]
    )

def run_feedback_source_trajectory(q0,qid,policy,feedback_source):
    # Search/evaluation always remain on PQ32.
    # feedback_source in {"PQ32","SQ8"} only selects the evidence source.
    q0 = normalize_vec(q0)
    q = q0.copy()
    qrel = QREL_MAP.get(str(qid),{})

    final_u = None
    for t in range(CONFIG["rounds"]+1):
        pq_scores,pq_ids = search(pq_index,q[None,:])
        pq_scores,pq_ids = pq_scores[0],pq_ids[0]
        valid = pq_ids>=0
        pq_scores_v = pq_scores[valid]
        pq_ids_v = pq_ids[valid]

        docs = [doc_ids[j] for j in pq_ids_v]
        final_u = ndcg_at_k(docs,qrel,CONFIG["utility_k"])

        if t==CONFIG["rounds"]:
            break

        if feedback_source=="PQ32":
            fb_scores = pq_scores_v
            fb_ids = pq_ids_v
        elif feedback_source=="SQ8":
            sq_scores,sq_ids = search(sq_index,q[None,:])
            sq_scores,sq_ids = sq_scores[0],sq_ids[0]
            sv = sq_ids>=0
            fb_scores = sq_scores[sv]
            fb_ids = sq_ids[sv]
        else:
            raise ValueError(feedback_source)

        k = int(policy["k"])
        if len(fb_ids)<k:
            raise RuntimeError("Too few feedback documents.")

        f = feedback_vector(
            get_corpus_rows(fb_ids[:k]),
            fb_scores[:k],
            policy
        )
        q = update_state(q0,f,policy["alpha"])

    return float(final_u)


## 12. Resumable E5 intervention runs

For each frozen policy and every untouched validation query, compute:

- final utility with PQ32-sourced feedback;
- final utility with SQ8-sourced feedback.

RASF and matched random allocations can then be evaluated exactly from the per-query feedback-source gain without rerunning trajectories for every budget/allocation.


In [ ]:
VAL_MASK = query_mask(query_ids,val_ids)
VAL_Q = np.asarray(queries[VAL_MASK],dtype=np.float32)
VAL_QIDS = query_ids[VAL_MASK]
assert len(VAL_QIDS)==3316

def run_policy_resumable(policy):
    pdir = OUT/policy["name"]
    pdir.mkdir(parents=True,exist_ok=True)
    chunk = CONFIG["checkpoint_every_queries"]

    for start in range(0,len(VAL_QIDS),chunk):
        stop = min(start+chunk,len(VAL_QIDS))
        cp = pdir/f"intervention_{start:04d}_{stop:04d}.parquet"
        if cp.exists():
            print("skip",policy["name"],cp.name)
            continue

        rows=[]
        t0=time.perf_counter()
        for i in range(start,stop):
            q0=VAL_Q[i]
            qid=VAL_QIDS[i]
            u_pq=run_feedback_source_trajectory(q0,qid,policy,"PQ32")
            u_sq=run_feedback_source_trajectory(q0,qid,policy,"SQ8")
            rows.append({
                "query_id":str(qid),
                "policy":policy["name"],
                "u_always_pq_feedback":u_pq,
                "u_always_sq_feedback":u_sq,
                "feedback_gain":u_sq-u_pq,
            })

        tmp=cp.with_suffix(".tmp.parquet")
        pd.DataFrame(rows).to_parquet(tmp,index=False)
        os.replace(tmp,cp)
        print(policy["name"],cp.name,f"{time.perf_counter()-t0:.1f}s")

    parts=sorted(pdir.glob("intervention_*.parquet"))
    df=pd.concat([pd.read_parquet(p) for p in parts],ignore_index=True)
    assert len(df)==3316
    assert df["query_id"].nunique()==3316
    merged=OUT/f"v021_{policy['name']}_validation_feedback_source_outcomes.parquet"
    df.to_parquet(merged,index=False)
    return df

INTERVENTION = {}
for policy in CONFIG["policies"]:
    INTERVENTION[policy["name"]] = run_policy_resumable(policy)

print("ALL E5 FEEDBACK-SOURCE TRAJECTORIES COMPLETE")


## 13. RASF versus exact matched-budget random expectation + 10,000 allocations

In [ ]:
def rasf_budget_analysis(policy, outcomes, risk_rows, budget, seed):
    z = outcomes.merge(
        risk_rows, on="query_id", how="left", validate="one_to_one"
    )
    if z["risk_score"].isna().any():
        raise ValueError("Missing risk score.")

    n=len(z)
    k=max(1,int(round(budget*n)))

    order=np.argsort(-z["risk_score"].to_numpy())
    selected=np.zeros(n,dtype=bool)
    selected[order[:k]]=True

    base=z["u_always_pq_feedback"].to_numpy(dtype=float)
    gain=z["feedback_gain"].to_numpy(dtype=float)

    rasf_util=float(np.mean(base + selected*gain))
    random_expectation=float(np.mean(base) + (k/n)*np.mean(gain))
    always_pq=float(np.mean(base))
    always_sq=float(np.mean(base+gain))

    # Fixed-size matched random allocations.
    rng=np.random.default_rng(seed)
    random_utils=np.empty(CONFIG["random_allocations"],dtype=float)
    for b in range(CONFIG["random_allocations"]):
        idx=rng.choice(n,size=k,replace=False)
        mask=np.zeros(n,dtype=bool)
        mask[idx]=True
        random_utils[b]=np.mean(base + mask*gain)

    # Monte Carlo one-sided p with +1 correction.
    p=(1+np.sum(random_utils>=rasf_util))/(CONFIG["random_allocations"]+1)

    full_benefit=always_sq-always_pq
    recovery=np.nan if full_benefit==0 else (rasf_util-always_pq)/full_benefit

    return {
        "policy":policy["name"],
        "budget":budget,
        "n_queries":n,
        "selected_queries":k,
        "always_pq":always_pq,
        "always_sq_feedback":always_sq,
        "rasf":rasf_util,
        "random_expectation":random_expectation,
        "delta_vs_random":rasf_util-random_expectation,
        "mc_p_one_sided":float(p),
        "random_97p5":float(np.quantile(random_utils,0.975)),
        "rasf_exceeds_random_97p5":bool(rasf_util>np.quantile(random_utils,0.975)),
        "recovery_of_always_sq_benefit":float(recovery) if np.isfinite(recovery) else None,
    }

rows=[]
for pi,policy in enumerate(CONFIG["policies"]):
    outcomes=INTERVENTION[policy["name"]]
    risks=RISK_BY_POLICY[policy["name"]]
    for bi,budget in enumerate(CONFIG["budgets"]):
        rows.append(
            rasf_budget_analysis(
                policy,outcomes,risks,budget,
                seed=CONFIG["seed"]+100*pi+bi
            )
        )

rasf_df=pd.DataFrame(rows)
rasf_df.to_csv(OUT/"v021_e5_rasf_budget_results.csv",index=False)
display(rasf_df)


## 14. Primary 25% replication gate

In [ ]:
primary = rasf_df[np.isclose(rasf_df["budget"],CONFIG["primary_budget"])].copy()

# Outcome-neutral gate fixed by prior RASF logic:
# both frozen policy families must exceed random expectation at 25%,
# and the 10k-allocation one-sided MC p must be <= 0.05.
primary["passes"] = (
    (primary["delta_vs_random"] > 0) &
    (primary["mc_p_one_sided"] <= 0.05)
)

if primary["passes"].all():
    replication_gate="E5_RASF_PRIMARY_REPLICATES_BOTH_POLICIES"
elif primary["passes"].any():
    replication_gate="E5_RASF_PRIMARY_PARTIAL_REPLICATION"
else:
    replication_gate="E5_RASF_PRIMARY_FAILS_TO_REPLICATE"

print("REPLICATION GATE:",replication_gate)
display(primary)


## 15. Final report and artifact hashes

In [ ]:
final_report = {
    "study_id": CONFIG["study_id"],
    "status": "COMPLETE_E5_RASF_CROSS_ENCODER_REPLICATION",
    "replication_gate": replication_gate,
    "predictor_validation": {
        **pred_point,
        "cluster_bootstrap_95ci": pred_ci,
    },
    "primary_budget_results": primary.to_dict(orient="records"),
    "all_budget_results": rasf_df.to_dict(orient="records"),
    "interpretation_constraints": [
        "This is a within-E5 FIT-to-validation method replication, not zero-shot transfer of BGE-trained model weights.",
        "Search and evaluation remain on PQ32; only the feedback source is selectively changed.",
        "The two intervention policies and the 25% primary budget are inherited from the prior BGE RASF closure.",
        "No FEVER test relevance data were accessed.",
    ],
    "source_hashes": {
        "v018_protocol_sha256": sha256_file(V018_PROTOCOL),
        "v018_manifest_sha256": sha256_file(V018_MANIFEST),
        "v018_fit_endpoints_sha256": sha256_file(V018_FIT_ENDPOINTS),
        "v018_validation_endpoints_sha256": sha256_file(V018_VAL_ENDPOINTS),
        "v021_protocol_sha256": sha256_file(PROTOCOL_PATH),
        "v021_model_freeze_sha256": sha256_file(MODEL_FREEZE_PATH),
    },
    "software": {
        "faiss": faiss.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "python": sys.version.split()[0],
    },
    "test_accessed": False,
    "test_relevance_accessed": False,
}

REPORT_PATH=OUT/"v021_e5_rasf_final_report.json"
REPORT_PATH.write_text(json.dumps(final_report,indent=2))

artifacts=[
    PROTOCOL_PATH,
    MODEL_FREEZE_PATH,
    FEATURE_PATH,
    OUT/"v021_e5_fit_risk_events.parquet",
    OUT/"v021_e5_validation_risk_events.parquet",
    OUT/"v021_e5_rasf_budget_results.csv",
    REPORT_PATH,
]
for policy in CONFIG["policies"]:
    artifacts.append(
        OUT/f"v021_{policy['name']}_validation_feedback_source_outcomes.parquet"
    )

hash_rows=[]
for p in artifacts:
    if p.exists():
        hash_rows.append({
            "file":p.name,
            "bytes":p.stat().st_size,
            "sha256":sha256_file(p),
        })
hash_df=pd.DataFrame(hash_rows)
hash_df.to_csv(OUT/"V021_ARTIFACT_SHA256.csv",index=False)

print(json.dumps(final_report,indent=2)[:10000])
display(hash_df)
print("FINAL REPORT:",REPORT_PATH)


# Paper-facing interpretation

The manuscript should describe this as a **cross-encoder replication of the RASF method**, not as zero-shot transfer.

If both 25% configurations pass:

> “Using the same deployable feature family, frozen intervention policies, and 25% primary budget, RASF was re-fit on E5 FIT and applied unchanged to untouched E5 validation. Risk-aware allocation again exceeded matched-budget random allocation for both feedback families.”

If only one passes:

> “The controller partially transfers across encoder regimes: one frozen feedback family retained a significant matched-budget advantage while the other did not.”

If neither passes:

> “The diagnostic phenomenon generalized more strongly than the controller; RASF did not retain its matched-budget advantage under E5.”

All three outcomes are scientifically admissible and must be retained.
